# 🔧 Spark Worker Setup — Google Colab

**Jalankan sel-sel berikut secara berurutan.**

> ⚠️ **JANGAN** menutup atau menghentikan Sel terakhir (Spark Worker) selama eksperimen berjalan.

> 📋 **Sebelum mulai:** Pastikan runtime sudah diset ke **GPU (T4)**
> Runtime → Change runtime type → GPU (T4)

In [ ]:
# ═══════════════════════════════════════════════════
# SEL 1: CEK ENVIRONMENT
# ═══════════════════════════════════════════════════
import sys
print(f'Python version: {sys.version}')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ═══════════════════════════════════════════════════
# SEL 2: INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════
# PENTING: Versi PySpark HARUS 3.5.6 (sama dengan Master/Laptop)
!pip install -q pyspark==3.5.6
!pip install -q transformers datasets accelerate evaluate peft scikit-learn tqdm

In [ ]:
# ═══════════════════════════════════════════════════
# SEL 3: INSTALL TAILSCALE
# ═══════════════════════════════════════════════════
!curl -fsSL https://tailscale.com/install.sh | sh

In [ ]:
# ═══════════════════════════════════════════════════
# SEL 4: CONNECT KE TAILSCALE (USERSPACE)
# ═══════════════════════════════════════════════════
!pkill tailscaled
import subprocess, time

# Mode userspace
subprocess.Popen(
    ["tailscaled", "--tun=userspace-networking", "--socks5-server=localhost:1055"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

AUTH_KEY     = "tskey-auth-GANTI_DENGAN_AUTH_KEY_MU"  # <-- GANTI INI DENGAN KEY KAMU
WORKER_NAME  = "colab-worker-1"                       # Ganti jadi colab-worker-2 untuk Incognito

!tailscale up --authkey="{AUTH_KEY}" --hostname="{WORKER_NAME}" --reset

In [ ]:
# ═══════════════════════════════════════════════════
# SEL 5: VERIFIKASI KONEKSI
# ═══════════════════════════════════════════════════
MASTER_IP = "100.70.125.49"  # IP Tailscale laptop

print("=== Tailscale IP ===")
!tailscale ip -4

print(f'\n=== Ping Master ({MASTER_IP}) ===')
print("Note: Ping ke Windows mungkin RTO karena Firewall. Abaikan jika Timeout.")
!tailscale ping {MASTER_IP}

In [ ]:
# ═══════════════════════════════════════════════════
# SEL 6: PRE-DOWNLOAD MODEL & DATASET
# ═══════════════════════════════════════════════════
# Cache model dan dataset SEBELUM Worker berjalan
# agar evaluasi fitness tidak perlu download ulang

import os, urllib.request

# Download dataset
os.makedirs("data/raw", exist_ok=True)
DATA_URL  = "https://raw.githubusercontent.com/Vuxyn/IndoStockSense/main/data/raw/Dataset-CNBCI-Sentimented.csv"
DATA_PATH = "data/raw/Dataset-CNBCI-Sentimented.csv"
if not os.path.exists(DATA_PATH):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f"✅ Dataset saved to {DATA_PATH}")
else:
    print("✅ Dataset already cached")

# Pre-cache IndoBERT model
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "indobenchmark/indobert-base-p1"
print("\nDownloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Downloading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True
)
del model  # Hapus dari memori GPU, sudah ter-cache di disk
print("✅ Model cached!")

## ⚠️ Sel Berikutnya Akan BLOCKING

Sel di bawah ini akan **terus berjalan** selama eksperimen berlangsung.

**JANGAN** menghentikan sel ini sampai eksperimen selesai!

In [ ]:
# ═══════════════════════════════════════════════════
# SEL 7: JALANKAN SPARK WORKER (FIX BINDING)
# ═══════════════════════════════════════════════════
import subprocess, os

# Cari path spark-class
result = subprocess.run(
    ["find", "/", "-name", "spark-class", "-path", "*/pyspark/*", "-not", "-name", "*.cmd"],
    capture_output=True, text=True, timeout=15
)
SPARK_CLASS = result.stdout.strip().split("\n")[0]

# Dapatkan IP Tailscale
result = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True)
WORKER_IP = result.stdout.strip()
MASTER_IP = "100.70.125.49"  # IP Tailscale laptop

print(f"Worker IP  : {WORKER_IP}")
print(f"Master URL : spark://{MASTER_IP}:7077")
print("\n🚀 Starting Spark Worker...")

# 1. Buat folder config khusus
CONF_DIR = "/tmp/spark-conf"
os.makedirs(CONF_DIR, exist_ok=True)

# 2. Tulis spark.bindAddress ke dalam spark-defaults.conf
# Ini akan memaksa Spark bind ke semua interface, menghindari BindException
with open(f"{CONF_DIR}/spark-defaults.conf", "w") as f:
    f.write("spark.bindAddress 0.0.0.0\n")

# 3. Beri tahu Spark di mana mencari file config
os.environ["SPARK_CONF_DIR"] = CONF_DIR

# 4. Tetap publikasikan IP Tailscale ke Master
os.environ["SPARK_LOCAL_HOSTNAME"] = WORKER_IP

# Jalankan Worker — BLOCKING
!{SPARK_CLASS} org.apache.spark.deploy.worker.Worker \
    spark://{MASTER_IP}:7077 \
    --host {WORKER_IP} \
    --cores 2 \
    --memory 10g